# TSA_ch12_smoothing

**Published in:** Time Series Analysis  
**Author:** Daniel Traian Pele  
**Description:** Smoothed periodogram using Daniell kernel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram

COLORS = {'blue':'#1A3A6E','red':'#DC3545','green':'#2E7D32','amber':'#B5853F','orange':'#E6802E','purple':'#8E44AD','gray':'#666666'}

In [ ]:
def daniell_smooth(psd, M):
    """Moving average of periodogram with half-bandwidth M."""
    kernel = np.ones(2*M+1) / (2*M+1)
    return np.convolve(psd, kernel, mode='same')

rng = np.random.default_rng(1)
phi = 0.6
N = 1024
eps = rng.normal(0, 1, N+100)
x = np.zeros(N+100)
for t in range(1, N+100):
    x[t] = phi*x[t-1] + eps[t]
x = x[100:]
true_psd = lambda f: 1.0 / (1 - 2*phi*np.cos(2*np.pi*f) + phi**2)

freqs, raw = periodogram(x)
f_true = np.linspace(0.01, 0.49, 400)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(freqs[1:], raw[1:], color=COLORS['gray'], lw=0.7, alpha=0.5, label='Raw periodogram')
for M, col in [(5, COLORS['blue']), (15, COLORS['orange']), (30, COLORS['red'])]:
    smoothed = daniell_smooth(raw, M)
    ax.semilogy(freqs[1:], smoothed[1:], color=col, lw=1.8, label=f'Daniell M={M}')
ax.semilogy(f_true, true_psd(f_true), 'k--', lw=1.5, label='True PSD')
ax.set_xlabel('Frequency')
ax.set_ylabel('PSD (log scale)')
ax.set_title('Daniell Kernel Smoothing — Bias-Variance Tradeoff')
fig.patch.set_alpha(0); ax.set_facecolor('none')
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=5, frameon=False)
plt.tight_layout()
plt.savefig('smoothing_comparison.pdf', bbox_inches='tight')
plt.show()